In [1]:
from langchain_openai import AzureChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, PydanticOutputParser
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

import os
load_dotenv()


c:\Users\rp00988241\AppData\Local\miniconda3\envs\genai-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
llm = AzureChatOpenAI(azure_deployment='gpt-4o')
llm.invoke("Hi Gemini")

AIMessage(content='Hello! How can I assist you today? 😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 9, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_b54fe76834', 'id': 'chatcmpl-CWKsgQwUDlqBBN2xMAoQZkcKZ86uk', 'service_tier': None, 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}], 'finish_reason': 'stop', 'logprobs': None, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 

In [4]:
# 1st prompt -> detailed report
template1 = PromptTemplate(
    template='Write in 500 words report on {topic}',
    input_variables=['topic']
)

# 2nd prompt -> summary
template2 = PromptTemplate(
    template='Write 3 pointers on the following text. /n {text}',
    input_variables=['text']
)

prompt1 = template1.invoke({'topic':'black hole'})

result = llm.invoke(prompt1)

prompt2 = template2.invoke({'text':result.content})

result1 = llm.invoke(prompt2)

print(result1.content)

1. **Formation and Types of Black Holes**: Black holes form when a massive star's nuclear fuel depletes, leading to gravitational collapse. Depending on their size, they can become stellar-mass black holes, intermediate black holes, or supermassive black holes, with the latter typically residing at the centers of galaxies.

2. **Characteristics and Detection**: Black holes are defined by their mass, spin, and charge. While invisible due to their lack of light emission, they can be observed indirectly through their gravitational impact, surrounding accretion disks emitting high-energy radiation, and phenomena such as gravitational lensing.

3. **Astrophysical and Theoretical Significance**: Black holes distort spacetime, exhibiting effects like gravitational lensing and time dilation. Observational milestones like the detection of gravitational waves (2015) and the first image of a black hole's event horizon (2019) have advanced our understanding of their nature, bridging gaps between g

In [11]:
parser = StrOutputParser()

chain = template1 | llm | parser | template2 | llm | parser

result = chain.invoke({'topic':'AI technonogy'})

print(result)


1. **Key Components and Types of AI**: Artificial Intelligence is divided into Narrow AI (used for specific tasks, like virtual assistants and recommendation systems) and General AI (envisioned to perform human-like versatile tasks). It relies on subfields like machine learning, deep learning, natural language processing, computer vision, and robotics to execute intelligent functions.

2. **Applications Across Industries**: AI is transforming industries such as healthcare (for diagnostics and drug discovery), finance (fraud detection and risk assessment), transportation (autonomous vehicles and logistics), retail (personalized experiences), manufacturing (predictive maintenance and quality control), education (personalized learning tools), and entertainment (content recommendations and automated creation).

3. **Challenges, Ethics, and the Future**: Ethical concerns like job displacement, data privacy, algorithmic bias, and generative AI misuse (e.g., deepfakes) pose challenges. The fu

In [9]:
parser.invoke(result1)

"1. **Formation and Types of Black Holes**: Black holes form when a massive star's nuclear fuel depletes, leading to gravitational collapse. Depending on their size, they can become stellar-mass black holes, intermediate black holes, or supermassive black holes, with the latter typically residing at the centers of galaxies.\n\n2. **Characteristics and Detection**: Black holes are defined by their mass, spin, and charge. While invisible due to their lack of light emission, they can be observed indirectly through their gravitational impact, surrounding accretion disks emitting high-energy radiation, and phenomena such as gravitational lensing.\n\n3. **Astrophysical and Theoretical Significance**: Black holes distort spacetime, exhibiting effects like gravitational lensing and time dilation. Observational milestones like the detection of gravitational waves (2015) and the first image of a black hole's event horizon (2019) have advanced our understanding of their nature, bridging gaps betw

In [ ]:
# Structured Output Parser
schema = [
    ResponseSchema(name='fact_1', description='Fact 1 about the topic'),
    ResponseSchema(name='fact_2', description='Fact 2 about the topic'),
    ResponseSchema(name='fact_3', description='Fact 3 about the topic'),
]

parser = StructuredOutputParser.from_response_schemas(schema)

template = PromptTemplate(
    template='Give 3 fact about {topic} \n {format_instruction}',
    input_variables=['topic'],
    partial_variables={'format_instruction':parser.get_format_instructions()}
)

chain = template | llm | parser

result = chain.invoke({'topic':'black hole'})

print(result)

{'fact_1': 'Black holes are regions in space where gravity is so strong that nothing, not even light, can escape from them.', 'fact_2': 'The boundary around a black hole beyond which nothing can escape is called the event horizon.', 'fact_3': 'Black holes can grow by accumulating matter and merging with other black holes, creating even larger black holes.'}


In [13]:
parser = JsonOutputParser()

template = PromptTemplate(
    template='Give me 5 facts about {topic} \n {format_instruction}',
    input_variables=['topic'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

chain = template | llm | parser

result = chain.invoke({'topic':'black hole'})

print(result)


{'facts': [{'fact': 'A black hole is a region of space with a gravitational pull so intense that nothing, not even light, can escape from it.'}, {'fact': 'Black holes are formed when massive stars collapse at the end of their life cycles, creating a point of infinite density called a singularity.'}, {'fact': 'The boundary surrounding a black hole, beyond which nothing can escape, is known as the event horizon.'}, {'fact': 'Black holes come in different sizes, including stellar black holes (from collapsing stars), supermassive black holes (found at the centers of galaxies), and intermediate black holes.'}, {'fact': 'The closest known black hole to Earth, V616 Monocerotis (or A0620-00), is about 3,000 light-years away.'}]}


In [14]:
from pydantic import BaseModel, Field

class Person(BaseModel):

    name: str = Field(description='Name of the person')
    age: int = Field(gt=18, description='Age of the person')
    city: str = Field(description='Name of the city the person belongs to')

parser = PydanticOutputParser(pydantic_object=Person)

template = PromptTemplate(
    template='Generate the name, age and city of a fictional {place} person\n {format_instruction}',
    input_variables=['place'],
    partial_variables={'format_instruction':parser.get_format_instructions()}
)

chain = template | llm | parser

final_result = chain.invoke({'place':'sri lankan'})

print(final_result)


name='Kasun Perera' age=29 city='Colombo'


In [15]:
final_result.age

29

In [16]:
template = PromptTemplate(
    template='Generate the name, age and city of a fictional {place} person',
    input_variables=['place']
)

chain = template | llm.with_structured_output(Person)
result = chain.invoke({'place':'sri lankan'})
result

Person(name='Anura Perera', age=34, city='Colombo')

In [32]:
result.age

28